# 05 · N-gram speculative decoding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/vllm/05_ngram_speculation.ipynb) &nbsp; 📖 Concept deck: [Speculative decoding in vLLM](https://nilesh-salpe.github.io/visual-ai/vllm/06-speculative-decoding/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** the n-gram ("prompt lookup") proposer vLLM ships, greedy verification of proposed tokens, and the expected tokens per step for a given acceptance rate.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


CODE = "def add(a, b): return a + b . def sub(a, b): return a - b . def mul(a, b): return a * b .".split()

## Task 1: Propose by matching the last n tokens

Find the most recent earlier occurrence of the last `n` tokens in the context (excluding the suffix itself) and propose the `k` tokens that followed it. Try n from `max_n` down to `min_n`; return `[]` if nothing matches. This is the idea behind vLLM's `method="ngram"` (prompt lookup decoding).

In [ ]:
def ngram_propose(tokens, k=3, max_n=3, min_n=1):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_ngram_propose(f):
    ctx = CODE[:CODE.index("sub(a,") + 2]       # ... "def", "sub(a,", "b):"
    assert f(ctx, k=3) == ["return", "a", "+"], "the earlier 'b):' was followed by 'return a +'"
    assert f(["x", "y", "z"], k=2) == [], "no earlier match"

check("ngram_propose", _test_ngram_propose, ngram_propose)

## Task 2: Verify against the target model

The target model scores all proposed positions in one forward pass. With greedy decoding, accept proposals left to right while they equal the target's own choice; at the first mismatch take the target's token instead. If all k are accepted, the pass also yields one bonus token. Return the list of tokens emitted this step (always ≥ 1).

In [ ]:
def verify_greedy(proposal, target_next):
    """target_next[i] = the target model's greedy token at proposal position i (len = len(proposal) + 1)."""
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_verify_greedy(f):
    assert f(["a", "b", "c"], ["a", "b", "c", "d"]) == ["a", "b", "c", "d"], "all accepted + bonus"
    assert f(["a", "x", "c"], ["a", "b", "c", "d"]) == ["a", "b"], "stop at first mismatch, emit the target's token"
    assert f([], ["z"]) == ["z"], "no proposal = ordinary decoding"

check("verify_greedy", _test_verify_greedy, verify_greedy)

## Task 3: Expected tokens per target pass

If each proposed token is accepted independently with probability α and k tokens are proposed, the expected number of tokens per target forward pass is (1 − α^(k+1)) / (1 − α) (Leviathan et al., 2023). Return it (handle α = 1).

In [ ]:
def expected_tokens(alpha, k):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_expected_tokens(f):
    assert abs(f(0.0, 4) - 1) < 1e-12 and abs(f(1.0, 4) - 5) < 1e-12
    assert abs(f(0.8, 4) - (1 - 0.8 ** 5) / 0.2) < 1e-12

check("expected_tokens", _test_expected_tokens, expected_tokens)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Propose by matching the last n tokens
def ngram_propose_solution(tokens, k=3, max_n=3, min_n=1):
    for n in range(max_n, min_n - 1, -1):
        if len(tokens) <= n: continue
        suffix = tokens[-n:]
        for s in range(len(tokens) - n - 1, -1, -1):
            if tokens[s:s + n] == suffix:
                return tokens[s + n:s + n + k]
    return []

check("ngram_propose (reference)", _test_ngram_propose, ngram_propose_solution)

In [ ]:
# @title Solution for Task 2: Verify against the target model
def verify_greedy_solution(proposal, target_next):
    out = []
    for p, t in zip(proposal, target_next):
        if p != t:
            return out + [t]
        out.append(p)
    return out + [target_next[len(proposal)]]

check("verify_greedy (reference)", _test_verify_greedy, verify_greedy_solution)

In [ ]:
# @title Solution for Task 3: Expected tokens per target pass
def expected_tokens_solution(alpha, k):
    return k + 1.0 if alpha == 1 else (1 - alpha ** (k + 1)) / (1 - alpha)

check("expected_tokens (reference)", _test_expected_tokens, expected_tokens_solution)

---
## Experiments
**Where n-gram speculation shines.** Repetitive outputs (code edits, quoting the prompt, structured data) give high acceptance. Simulate decoding `CODE` with a "target" that always knows the next token and count target passes.

In [ ]:
for k in [1, 3, 5]:
    ctx, passes = CODE[:8], 0
    while len(ctx) < len(CODE):
        prop = ngram_propose_solution(ctx, k=k)
        target = CODE[len(ctx):len(ctx) + len(prop) + 1] + ["<eos>"] * (len(prop) + 1)
        emitted = verify_greedy_solution(prop, target[:len(prop) + 1]); ctx += emitted; passes += 1
    print(f"k={k}: {len(CODE) - 8} tokens in {passes} target passes ({(len(CODE) - 8) / passes:.2f} tokens/pass)")
for a in [0.5, 0.7, 0.9]:
    print(f"alpha={a}: expected tokens/pass with k=4: {expected_tokens_solution(a, 4):.2f}")

---
## Takeaways
* Speculative decoding lets a cheap proposer guess k tokens and the target verify them in one pass; the output is identical to the target's own.
* vLLM's n-gram method needs no draft model: it copies continuations of earlier matching n-grams from the context.
* Speed-up depends on acceptance: (1 − α^(k+1)) / (1 − α) tokens per pass, minus the proposer and verification overhead.